# 04 — Cross-architecture comparison

A cross-architecture claim requires explicit Raspberry Pi 5 and x86 Linux inputs. If either half is absent, the output remains PENDING. N is runs, units are ratios of run-median latency, and evidence status is explicit.


In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, passed_artifacts, pending_record, percentile_rows
from wafer_analysis.paths import resolve_analysis_batch

def resolve(experiment, env_name):
    return resolve_analysis_batch(experiment, os.environ.get(env_name))[0]

rpi=resolve('e-perf-5','E_PERF_5_RPI_DIR')
x86=resolve('e-perf-5','E_PERF_5_X86_DIR') if os.environ.get('E_PERF_5_X86_DIR') else None
rows=[]
for architecture,batch in [('arm64-rpi5',rpi),('x86-linux',x86)]:
    df=pd.DataFrame() if batch is None else percentile_rows(batch)
    wafer=df[df.condition=='wafer'] if not df.empty else pd.DataFrame()
    native=df[df.condition=='native'] if not df.empty else pd.DataFrame()
    if wafer.empty or native.empty:
        rows.append(pending_record(architecture,'matched WAFER/native leaves are incomplete','WAFER/native p50 ratio'))
    else:
        rows.append({'question':architecture,'status':'READY','value':wafer.p50_ns.median()/native.p50_ns.median(),'units':'WAFER/native p50 ratio','N':f'{len(wafer)}+{len(native)} runs','uncertainty':'descriptive only','thesis_evidence':False})
out=pd.DataFrame(rows); display(out)
ready=out[out.status=='READY']
if len(ready)==2:
    ax=ready.plot.bar(x='question',y='value',legend=False); ax.set_ylabel('WAFER/native p50 ratio'); ax.set_title('Cross-architecture ratio — diagnostic')
else: print('No cross-architecture claim: both explicitly identified architecture batches are required.')
